## L08

In [19]:
import pandas as pd
import numpy as np
# --- Tabela 1: autorzy ---
autorzy = pd.DataFrame({
'autor_id': [1, 2, 3, 4, 5, 6, 7, 8],
'imie': ['Olga', 'Stanisław', 'Andrzej', 'Wisława', 'Ryszard',
'Dorota', 'Szczepan', 'Jacek'],
'nazwisko': ['Tokarczuk', 'Lem', 'Sapkowski', 'Szymborska', 'Kapuściński',
'Masłowska', 'Twardoch', 'Dehnel'],
'kraj': ['Polska', 'Polska', 'Polska', 'Polska', 'Polska',
'Polska', 'Polska', 'Polska'],
'nagrody': ['Nobel', 'SFF', 'SFF', 'Nobel', 'Reporter',
'Polityka', 'NIKE', 'NIKE']
})
# --- Tabela 2: książki ---
ksiazki = pd.DataFrame({
'ksiazka_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
'autor_id': [1, 1, 2, 2, 3, 3, 4, 5, 6, 7, 7, 8],
'tytul': ['Księgi Jakubowe', 'Bieguni', 'Solaris', 'Cyberiada',
'Wiedźmin', 'Narrenturm', 'Wiersze wybrane', 'Podróże z Herodotem',
'Wojna polsko-ruska', 'Morfina', 'Król', 'Lala'],
'kategoria': ['historyczna', 'obyczajowa', 'sci-fi', 'sci-fi',
'fantasy', 'fantasy', 'poezja', 'reportaz',
'obyczajowa', 'historyczna', 'historyczna', 'obyczajowa'],
'cena': [79.90, 49.00, 39.00, 42.00, 45.00, 55.00, 35.00,
52.00, 38.00, 48.00, 59.00, 44.00],
'strony': [912, 376, 198, 295, 320, 512, 180, 263, 153, 618, 688, 432]
})

# --- Tabela 3: zamówienia ---
np.random.seed(2026)
n = 80
zamowienia = pd.DataFrame({
'zam_id': range(5001, 5001 + n),
'ksiazka_id': np.random.choice(ksiazki['ksiazka_id'], size=n),
'ilosc': np.random.randint(1, 6, size=n),
'data': pd.to_datetime('2026-01-01') +
pd.to_timedelta(np.random.randint(0, 120, n), unit='D'),
'kanal': np.random.choice(['web', 'aplikacja', 'telefon'], size=n, p=
[0.6, 0.3, 0.1]),
'miasto': np.random.choice(['Warszawa', 'Kraków', 'Wrocław', 'Gdańsk',
'Poznań'], size=n)
})
print(f"Autorzy: {autorzy.shape}")
print(f"Ksiazki: {ksiazki.shape}")
print(f"Zamowienia: {zamowienia.shape}")

Autorzy: (8, 5)
Ksiazki: (12, 6)
Zamowienia: (80, 6)


## Ćwiczenie 1: merge() — łączenie dwóch tabel

In [20]:
# Połącz ksiazki + autorzy po kluczu autor_id
ksiazki_z_autorami = ksiazki.merge(autorzy, on='autor_id')
print(f"Shape: {ksiazki_z_autorami.shape}")
ksiazki_z_autorami.head()

Shape: (12, 10)


,ksiazka_id,autor_id,tytul,kategoria,cena,strony,imie,nazwisko,kraj,nagrody
0,101,1,Księgi Jakubowe,historyczna,79.9,912,Olga,Tokarczuk,Polska,Nobel
1,102,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel
2,103,2,Solaris,sci-fi,39.0,198,Stanisław,Lem,Polska,SFF
3,104,2,Cyberiada,sci-fi,42.0,295,Stanisław,Lem,Polska,SFF
4,105,3,Wiedźmin,fantasy,45.0,320,Andrzej,Sapkowski,Polska,SFF


# 1a

Wynik ma 12 wierszy i 10 kolumn.
Do tabeli dołączono: imię, nazwisko, kraj, nagrody. 


In [21]:
ksiazki_test = pd.concat([
ksiazki,
pd.DataFrame({'ksiazka_id':[999], 'autor_id':[999],
'tytul':['Ksiazka bez autora'], 'kategoria':['test'],
'cena':[30.0], 'strony':[100]})
], ignore_index=True)

## Zadanie 1b

In [22]:
inner = ksiazki_test.merge(autorzy, on='autor_id', how='inner')
left = ksiazki_test.merge(autorzy, on='autor_id', how='left')
right = ksiazki_test.merge(autorzy, on='autor_id', how='right')
outer = ksiazki_test.merge(autorzy, on='autor_id', how='outer')
print(f"inner: {inner.shape[0]} wierszy")
print(f"left: {left.shape[0]} wierszy")
print(f"right: {right.shape[0]} wierszy")
print(f"outer: {outer.shape[0]} wierszy")

inner: 12 wierszy
left: 13 wierszy
right: 12 wierszy
outer: 13 wierszy


In [23]:
audyt = ksiazki_test.merge(autorzy, on='autor_id', how='outer', indicator=True)
audyt['_merge'].value_counts()

_merge
both          12
left_only      1
right_only     0
Name: count, dtype: int64

# Zadanie 1c: W komórce Markdown odpowiedz: dlaczego inner ma mniej wierszy niż left?

Inner zachowuje tylko rekordy których klucze znajduja sie w obu tabelach jednocześnie.

Ponieważ w tabeli ksiazki_test dodaliśmy książkę o autor_id = 999, dla którego nie ma odpowiednika w tabeli autorzy, typ inner pomija ten wiersz, natomiast left go zachowuje, uzupełniając brakujące dane autora wartościami NaN.

## Ćwiczenie 2: Merge łańcuchowy + kolumny wyliczane

In [24]:
# Połącz: zamowienia + ksiazki + autorzy
pelne = (
zamowienia
.merge(ksiazki, on='ksiazka_id')
.merge(autorzy, on='autor_id')
)
print(f"Pelna tabela: {pelne.shape}")
pelne.head()

Pelna tabela: (80, 15)


,zam_id,ksiazka_id,ilosc,data,kanal,miasto,autor_id,tytul,kategoria,cena,strony,imie,nazwisko,kraj,nagrody
0,5001,102,5,2026-04-05,web,Poznań,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel
1,5002,107,2,2026-02-08,web,Poznań,4,Wiersze wybrane,poezja,35.0,180,Wisława,Szymborska,Polska,Nobel
2,5003,111,1,2026-04-19,web,Poznań,7,Król,historyczna,59.0,688,Szczepan,Twardoch,Polska,NIKE
3,5004,109,4,2026-02-24,web,Warszawa,6,Wojna polsko-ruska,obyczajowa,38.0,153,Dorota,Masłowska,Polska,Polityka
4,5005,105,2,2026-01-10,web,Warszawa,3,Wiedźmin,fantasy,45.0,320,Andrzej,Sapkowski,Polska,SFF


# Zadanie 2a: Dodaj cztery nowe kolumny:

In [25]:
# 1. Wartość zamówienia
pelne['wartosc'] = pelne['ilosc'] * pelne['cena']
# 2. Miesiąc zamówienia
pelne['miesiac'] = pelne['data'].dt.month
# 3. Pełne imię i nazwisko autora
pelne['autor_pelne'] = pelne['imie'] + ' ' + pelne['nazwisko']
# 4. Kategoria cenowa książki
pelne['kategoria_cenowa'] = np.where(pelne['cena'] >= 50, 'droga', 'tania')

# Zadanie 2b

In [26]:
# 1. Ile jest wszystkich zamówień?
liczba_zamowien = len(pelne)
# Alternatywnie: pelne['zam_id'].count()

# 2. Jaki jest łączny przychód?
laczny_przychod = pelne['wartosc'].sum()

# 3. Jaka jest średnia wartość zamówienia?
srednia_wartosc = pelne['wartosc'].mean()

# 4. Ile książek (unikalnych tytułów) się sprzedało?
unikalne_ksiazki = pelne['tytul'].nunique()

print(f"Liczba zamówień: {liczba_zamowien}")
print(f"Łączny przychód: {laczny_przychod:.2f} zł")
print(f"Średnia wartość zamówienia: {srednia_wartosc:.2f} zł")
print(f"Liczba sprzedanych unikalnych tytułów: {unikalne_ksiazki}")

Liczba zamówień: 80
Łączny przychód: 11963.80 zł
Średnia wartość zamówienia: 149.55 zł
Liczba sprzedanych unikalnych tytułów: 12


## Ćwiczenie 3: groupby + .agg() — raporty per kategoria

# Zadanie 3a

In [27]:
raport_kategorie = pelne.groupby('kategoria')['wartosc'].sum()

raport_kategorie = raport_kategorie.sort_values(ascending=False)

print("Łączny przychód per kategoria:")
print(raport_kategorie)

Łączny przychód per kategoria:
kategoria
historyczna    4136.8
sci-fi         2334.0
obyczajowa     1999.0
fantasy        1930.0
reportaz       1144.0
poezja          420.0
Name: wartosc, dtype: float64


# Zadanie 3b

In [28]:
raport_rozszerzony = pelne.groupby('kategoria')['wartosc'].agg(['count', 'mean', 'sum'])

raport_rozszerzony = raport_rozszerzony.round(2)

raport_rozszerzony = raport_rozszerzony.sort_values('sum', ascending=False)

print("Statystyki sprzedaży per kategoria:")
print(raport_rozszerzony)

Statystyki sprzedaży per kategoria:
             count    mean     sum
kategoria                         
historyczna     22  188.04  4136.8
sci-fi          15  155.60  2334.0
obyczajowa      17  117.59  1999.0
fantasy         14  137.86  1930.0
reportaz         5  228.80  1144.0
poezja           7   60.00   420.0


# Zadanie 3c

In [29]:
raport_autorzy = pelne.groupby('autor_pelne').agg(
    liczba_zamowien=('zam_id', 'count'),      
    laczna_sprzedaz=('wartosc', 'sum'),       
    sredni_ilosc=('ilosc', 'mean')            
)

raport_autorzy = raport_autorzy.round(2).sort_values('laczna_sprzedaz', ascending=False)

print("Ranking autorów wg wartości sprzedaży:")
print(raport_autorzy)

Ranking autorów wg wartości sprzedaży:
                     liczba_zamowien  laczna_sprzedaz  sredni_ilosc
autor_pelne                                                        
Olga Tokarczuk                    18           3389.8          2.72
Stanisław Lem                     15           2334.0          3.80
Andrzej Sapkowski                 14           1930.0          2.86
Szczepan Twardoch                 11           1580.0          2.91
Ryszard Kapuściński                5           1144.0          4.40
Jacek Dehnel                       5            748.0          3.40
Wisława Szymborska                 7            420.0          1.71
Dorota Masłowska                   5            418.0          2.20


# Zadanie 3d

In [30]:
raport_kat_kanal = pelne.groupby(['kategoria', 'kanal'])['wartosc'].sum()

print("Przychód według kategorii i kanału sprzedaży:")
print(raport_kat_kanal)

Przychód według kategorii i kanału sprzedaży:
kategoria    kanal    
fantasy      aplikacja     475.0
             telefon       225.0
             web          1230.0
historyczna  aplikacja    1988.5
             telefon       527.4
             web          1620.9
obyczajowa   aplikacja     586.0
             telefon       136.0
             web          1277.0
poezja       aplikacja      70.0
             telefon        70.0
             web           280.0
reportaz     aplikacja     416.0
             telefon       260.0
             web           468.0
sci-fi       aplikacja     534.0
             telefon       279.0
             web          1521.0
Name: wartosc, dtype: float64


# Zadanie 3e

In [31]:
pelne['wartosc_kat'] = pelne.groupby('kategoria')['wartosc'].transform('sum')


pelne['udzial_w_kategorii_pct'] = (pelne['wartosc'] / pelne['wartosc_kat'] * 100).round(2)


print(pelne[['zam_id', 'kategoria', 'wartosc', 'udzial_w_kategorii_pct']].head())

   zam_id    kategoria  wartosc  udzial_w_kategorii_pct
0    5001   obyczajowa    245.0                   12.26
1    5002       poezja     70.0                   16.67
2    5003  historyczna     59.0                    1.43
3    5004   obyczajowa    152.0                    7.60
4    5005      fantasy     90.0                    4.66


# Ćwiczenie 4: pivot_table + crosstab

# Zadanie 4a

In [32]:
raport_2d = pd.pivot_table(
    pelne, 
    index='kategoria',     
    columns='miesiac',     
    values='wartosc',      
    aggfunc='sum',         
    fill_value=0           
)

print("Miesięczny przychód per kategoria:")
display(raport_2d)

Miesięczny przychód per kategoria:


miesiac,1,2,3,4
kategoria,,,,
fantasy,595.0,410.0,270.0,655.0
historyczna,1342.5,1790.9,634.4,369.0
obyczajowa,301.0,623.0,274.0,801.0
poezja,105.0,70.0,140.0,105.0
reportaz,416.0,0.0,728.0,0.0
sci-fi,765.0,615.0,672.0,282.0


# Zadanie 4b

In [33]:
raport_final = pd.pivot_table(
    pelne, 
    index='kategoria', 
    columns='miesiac', 
    values='wartosc', 
    aggfunc='sum', 
    fill_value=0,
    margins=True,              
    margins_name='RAZEM'        
)


print("Raport sprzedaży z podsumowaniem:")
display(raport_final)

Raport sprzedaży z podsumowaniem:


miesiac,1,2,3,4,RAZEM
kategoria,,,,,
fantasy,595.0,410.0,270.0,655.0,1930.0
historyczna,1342.5,1790.9,634.4,369.0,4136.8
obyczajowa,301.0,623.0,274.0,801.0,1999.0
poezja,105.0,70.0,140.0,105.0,420.0
reportaz,416.0,0.0,728.0,0.0,1144.0
sci-fi,765.0,615.0,672.0,282.0,2334.0
RAZEM,3524.5,3508.9,2718.4,2212.0,11963.8


# Zadanie 4c

In [34]:
rozklad_zamowien = pd.crosstab(
    pelne['kanal'],   
    pelne['miasto']   
)

print("Liczba zamówień per kanał i miasto:")
display(rozklad_zamowien)

Liczba zamówień per kanał i miasto:


miasto,Gdańsk,Kraków,Poznań,Warszawa,Wrocław
kanal,,,,,
aplikacja,7,1,5,6,4
telefon,1,4,1,3,2
web,5,9,14,9,9


# Zadanie 4d

In [35]:
rozklad_pct = pd.crosstab(
    pelne['kanal'], 
    pelne['miasto'], 
    normalize='index'  
) * 100              

rozklad_pct = rozklad_pct.round(2)

print("Udział miast w poszczególnych kanałach sprzedaży (%):")
display(rozklad_pct)

Udział miast w poszczególnych kanałach sprzedaży (%):


miasto,Gdańsk,Kraków,Poznań,Warszawa,Wrocław
kanal,,,,,
aplikacja,30.43,4.35,21.74,26.09,17.39
telefon,9.09,36.36,9.09,27.27,18.18
web,10.87,19.57,30.43,19.57,19.57


# Zadanie 4e 

Największy przychód generuje kategoria historyczna - 4136.80 zł, co stanowi ponad 1/3 całkowitej sprzedaży sklepu, natomiast najmniejszy poezja.

W badanym okresie obserwujemy sukcesywny spadek ogólnego przychodu w kolejnych miesiącach – od szczytowego poziomu 3524.50 zł w styczniu do 2212.00 zł w kwietniu.

Mimo ogólnego spadku rynku w kwietniu (4), kategorie fantasy oraz obyczajowa zanotowały w tym czasie swoje miesięczne rekordy sprzedaży.

Najpopularniejszym kanałem jest zazwyczaj web lub aplikacja. Kanał telefoniczny generuje najmniejszy ruch.





Co by powiedział kierownik sprzedaży?

Całkowity przychód wyniósł 11 963.80 zł, przy czym najwyższą rentowność osiągnęliśmy w pierwszym kwartale. Średnia wartość zamówienia utrzymuje się na stabilnym poziomie, jednak spadek łącznych wpływów w marcu i kwietniu o ok. 30% względem początku roku sugeruje potrzebę wdrożenia agresywniejszej polityki rabatowej na maj, aby odwrócić ten trend.